# 02 — Build the fight-level dataset

Turns the raw Kaggle tables into one clean, chronological row per decided
fight, with the fighter-ordering leakage found in `ufc_event_fight_stats.csv`
removed. 

All the actual transform logic lives in `src/ufc_predictor/data/` — this
notebook just calls it and checks the result. That split is deliberate: the
same `build_fight_dataset()` function is what a future incremental refresh
(new fights appended to `data/raw`) would call too, so the logic can't drift
between "what the notebook does" and "what the pipeline does".

In [ ]:
from ufc_predictor.data.build import build_fight_dataset
from ufc_predictor.data.io import save_processed

fights = build_fight_dataset()

## Sanity checks

The things that would silently break the project if they were wrong.

In [ ]:
# 7,840 raw fights minus 143 draws/no-contests
assert len(fights) == 7697
assert fights["event_date"].is_monotonic_increasing

overall_rate = fights["fighter_a_won"].mean()
print(f"fighter_a win rate, overall: {overall_rate:.3f}")
assert 0.45 < overall_rate < 0.55

In [ ]:
# the bug this notebook exists to fix: pre-2010, the raw data lists the
# winner as f1 for every single fight. after de-biasing, that block should
# look like any other era, not a flat 100%.
pre_2010 = fights[fights["event_date"].dt.year < 2010]
pre_2010_rate = pre_2010["fighter_a_won"].mean()
print(f"fighter_a win rate, pre-2010 ({len(pre_2010)} fights): {pre_2010_rate:.3f}")
assert 0.40 < pre_2010_rate < 0.60

In [ ]:
fights.groupby(fights["event_date"].dt.year)["fighter_a_won"].mean().plot(
    title="fighter_a win rate by year (should hover near 0.5 throughout)",
    ylim=(0, 1),
)

## Save

In [ ]:
save_processed(fights, "fights")
fights.head()